### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

BLOG : 
https://medium.com/@kbdhunga/advanced-rag-decomposition-technique-in-langchain-c0959541cfec

In [1]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# step1 : Load and split the dataset
loader = TextLoader("langchain-crewai-dataset.txt")
raw_docs = loader.load()

# split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain-crewai-dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain-crewai-dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain-crewai-dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# step 2: Embedding
embedding_model = OpenAIEmbeddings()

# vectorstore
vectorstore = FAISS.from_documents(chunks, embedding_model)

# step 3: MMR Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "lambda_mult": 0.7}
)

retriever

c:\Users\heman\Desktop\05 Ultimate RAG Bootcamp Using Langchain,LangGraph and Langsmith by Krish Naik\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A20404BA10>, search_type='mmr', search_kwargs={'k': 4, 'lambda_mult': 0.7})

In [3]:
# step 4 : LLM 
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
llm


ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002A20404ABA0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002A204049E80>, root_client=<openai.OpenAI object at 0x000002A204079E50>, root_async_client=<openai.AsyncOpenAI object at 0x000002A20407AD50>, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [4]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


# Step 5: prompt for Query decomposition
decomposition_prompt = PromptTemplate.from_template(
"""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:

"""
)


decomposition_chain = decomposition_prompt | llm | StrOutputParser()

decomposition_chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.\n\nQuestion: "{question}"\n\nSub-questions:\n\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002A20404ABA0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002A204049E80>, root_client=<openai.OpenAI object at 0x000002A204079E50>, root_async_client=<openai.AsyncOpenAI object at 0x000002A20407AD50>, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [5]:
# Example Test query decomposition
query = {"question": "How does LangChain use memory and agents compared to CrewAI?"}

result = decomposition_chain.invoke(query)
print(result)

1. How does LangChain utilize memory in its system?
2. How does LangChain utilize agents in its system?
3. How does CrewAI utilize memory in its system?
4. How does CrewAI utilize agents in its system?


# RAG Pipeline

In [6]:
from langchain.chains.combine_documents import create_stuff_documents_chain

# Step 6: QA chain per sub-question
qa_prompt = PromptTemplate.from_template(
"""
Use the context below to answer the question.

Context: {context}
Question: {input}
"""
)

qa_chain = create_stuff_documents_chain(
    llm=llm, 
    prompt=qa_prompt
)

qa_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nUse the context below to answer the question.\n\nContext: {context}\nQuestion: {input}\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000002A20404ABA0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002A204049E80>, root_client=<openai.OpenAI object at 0x000002A204079E50>, root_async_client=<openai.AsyncOpenAI object at 0x000002A20407AD50>, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [7]:
# Step 7: Full RAG pipeline logic
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]
    
    results = []
    
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        result = qa_chain.invoke({"input": subq, "context": docs})
        results.append(f"Q: {subq}\nA: {result}")
    
    return "\n\n".join(results)

In [8]:
# Step 8: Run
query = "How does LangChain use memory and agents compared to CrewAI?"

final_answer = full_query_decomposition_rag_pipeline(query)

print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: What is the role of memory in LangChain's system?
A: The role of memory in LangChain's system is to allow the Language Model (LLM) to maintain awareness of previous conversation turns and summarize long interactions to fit within token limits.

Q: How does LangChain utilize agents within its platform?
A: LangChain utilizes agents within its platform by allowing them to use LLMs to reason about which tools to call, what input to provide, and how to process the output. Agents can execute multi-step tasks and integrate with tools like web search, calculators, and code execution. Developers can create pipelines that connect LLMs with various tools, APIs, vector databases, and other knowledge sources.

Q: What is the significance of memory within CrewAI's system?
A: The significance of memory within CrewAI's system is that developers can specify memory within a crew's configuration. This allows agents within the crew to remember past interactions, decisions, and informat